# RelCheck Spatial Detection Test

Real pipeline test:
1. Load image + caption
2. Extract entities from caption (Stage 1)
3. Run GDINO detection on those entities (Stage 2b)
4. Measure: what % of entities were detected?


In [ ]:
!pip install -q 'transformers>=4.50.0' torch torchvision pillow

import torch
import json
import re
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# Load GDINO
gdino_id = 'IDEA-Research/grounding-dino-tiny'
gdino_proc = AutoProcessor.from_pretrained(gdino_id)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(gdino_id).to(DEVICE)
print('✓ GroundingDINO loaded')

GDINO_BOX_THRESHOLD = 0.15
GDINO_TEXT_THRESHOLD = 0.10


In [ ]:
# Stage 1: Triple extraction (simplified for testing)
# Extract nouns from caption using simple rules

def extract_nouns(caption: str) -> list:
    '''Extract noun entities from caption.'''
    # Simple approach: split by spaces, filter by common patterns
    # In real pipeline this uses Llama-3.3-70B
    words = caption.lower().split()
    
    # Remove articles and common words
    stops = {'a', 'an', 'the', 'is', 'are', 'was', 'were', 'on', 'in', 'at',
             'to', 'from', 'with', 'by', 'and', 'or', 'but', 'of', 'for'}
    
    nouns = [w.strip('.,!?;:') for w in words if w not in stops and len(w) > 2]
    return list(dict.fromkeys(nouns))  # deduplicate

# Test extraction
test_caption = 'A cat is sitting on the couch next to a person holding a remote control.'
entities = extract_nouns(test_caption)
print(f'Caption: {test_caption}')
print(f'Extracted entities: {entities}')


In [ ]:
# Stage 2b: Run GDINO on extracted entities (batched)

def run_gdino_batched(img: Image.Image, queries: list, batch_size: int = 4) -> dict:
    '''Run GDINO detection in batches.'''
    if not queries:
        return {}
    
    clean_queries = list(dict.fromkeys(q.strip().lower() for q in queries if q.strip()))
    
    # Add articles
    article_queries = []
    article_to_query = {}
    for q in clean_queries:
        if q.split()[0] in ('a', 'an', 'the'):
            aq = q
        else:
            aq = f'a {q}'
        article_queries.append(aq)
        article_to_query[aq] = q
    
    results = {q: [] for q in clean_queries}
    
    # Process in batches
    for batch_start in range(0, len(article_queries), batch_size):
        batch = article_queries[batch_start:batch_start + batch_size]
        batch_map = {aq: article_to_query[aq] for aq in batch}
        
        inputs = gdino_proc(
            images=img,
            text=[batch],
            return_tensors='pt',
        ).to(DEVICE)
        
        with torch.no_grad():
            outputs = gdino_model(**inputs)
        
        results_list = gdino_proc.post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            threshold=GDINO_BOX_THRESHOLD,
            text_threshold=GDINO_TEXT_THRESHOLD,
            target_sizes=[(img.height, img.width)],
        )
        
        raw = results_list[0]
        boxes = raw['boxes'].cpu().numpy()
        scores = raw['scores'].cpu().numpy()
        
        if 'text_labels' in raw:
            labels = raw['text_labels']
        else:
            labels = raw.get('labels', [])
            if hasattr(labels, 'cpu'):
                labels = labels.cpu().tolist()
        
        for box, score, label in zip(boxes, scores, labels):
            x1, y1, x2, y2 = box.tolist()
            x1, y1 = max(0, int(x1)), max(0, int(y1))
            x2, y2 = min(img.width, int(x2)), min(img.height, int(y2))
            
            label_str = (label if isinstance(label, str) else str(label)).strip().lower()
            
            matched_query = batch_map.get(label_str)
            if matched_query is None:
                stripped = re.sub(r'^(a|an|the)\s+', '', label_str)
                for aq, oq in batch_map.items():
                    if oq == stripped or oq in stripped or stripped in oq:
                        matched_query = oq
                        break
            if matched_query is None:
                for q in [article_to_query[aq] for aq in batch]:
                    if q in label_str or label_str in q:
                        matched_query = q
                        break
            if matched_query is None:
                matched_query = label_str
            
            if matched_query not in results:
                results[matched_query] = []
            
            results[matched_query].append({
                'box': [x1, y1, x2, y2],
                'score': float(score),
                'label': matched_query,
            })
    
    return results

print('✓ run_gdino_batched defined')


In [ ]:
# Download the official HuggingFace example image (has cat + remote)
import requests
from io import BytesIO

url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
print(f'Downloading test image from {url}...')
img = Image.open(BytesIO(requests.get(url, timeout=10).content)).convert('RGB')
print(f'✓ Image loaded: {img.size[0]}x{img.size[1]}px')

# Display
plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.title('Test Image')
plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Test 1: With proper caption
print('='*70)
print('TEST 1: Real pipeline with caption')
print('='*70)

caption = 'A cat is sitting on the remote control.'
print(f'\nCaption: {caption}')

# Extract entities
entities = extract_nouns(caption)
print(f'Extracted entities: {entities}')

# Run GDINO
detections = run_gdino_batched(img, entities)

# Measure
detected = {q: d for q, d in detections.items() if d}
detection_rate = len(detected) / len(entities) if entities else 0

print(f'\nDetection results:')
for q in entities:
    dets = detections[q]
    if dets:
        print(f'  ✓ {q:15s}: {len(dets)} detection(s), best score {max(d["score"] for d in dets):.3f}')
    else:
        print(f'  ✗ {q:15s}: NOT DETECTED')

print(f'\nDetection rate: {len(detected)}/{len(entities)} = {detection_rate*100:.1f}%')

# Visualize
if detected:
    annotated = img.copy()
    draw = ImageDraw.Draw(annotated)
    colors = ['red', 'blue', 'green', 'yellow', 'purple']
    for ci, (q, dets) in enumerate(detected.items()):
        color = colors[ci % len(colors)]
        for d in dets:
            x1, y1, x2, y2 = d['box']
            draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
            draw.text((x1, y1-10), f"{q} ({d['score']:.2f})", fill=color)
    
    plt.figure(figsize=(10, 8))
    plt.imshow(annotated)
    plt.title(f'Detections: {len(detected)}/{len(entities)} entities')
    plt.axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
# Test 2: Multiple captions to get metrics
print('='*70)
print('TEST 2: Multiple captions - detection metrics')
print('='*70)

test_captions = [
    'A cat sits on the table.',
    'A person holds a remote control.',
    'The cat is near the couch.',
    'A cat and person in the room.',
]

results_summary = []
total_entities = 0
total_detected = 0

for caption in test_captions:
    entities = extract_nouns(caption)
    detections = run_gdino_batched(img, entities)
    detected = {q: d for q, d in detections.items() if d}
    
    detection_rate = len(detected) / len(entities) if entities else 0
    total_entities += len(entities)
    total_detected += len(detected)
    
    results_summary.append({
        'caption': caption,
        'entities': len(entities),
        'detected': len(detected),
        'rate': detection_rate,
    })
    
    print(f'{caption:<50s} | {len(detected)}/{len(entities)} ({detection_rate*100:5.1f}%)')

print('-'*70)
overall_rate = total_detected / total_entities if total_entities else 0
print(f'OVERALL: {total_detected}/{total_entities} = {overall_rate*100:.1f}%')
